# 06. Ревизия исправлений разметки

Ноутбук показывает 172 предложения по исправлению вместе с контекстом. В каноническую версию попадают только строки со статусом `ACCEPTED`. После изменения уже собранного датасета создавайте новую версию (`corrected_v2`), не перезаписывайте `corrected_v1`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import runpy
PROJECT_DIR = Path('/content/drive/MyDrive/NER_RuREBus_project')
BOOTSTRAP = PROJECT_DIR / 'colab_bootstrap.py'
DECISIONS = PROJECT_DIR / 'configs/data/corrections/rurebus_corrected_v1.csv'
REGISTRY = PROJECT_DIR / 'results/data_audit/rurebus_conflict_resolution_v1/surface_conflict_registry.csv'
for path in (BOOTSTRAP, DECISIONS, REGISTRY):
    if not path.is_file():
        raise FileNotFoundError(path)
runpy.run_path(str(BOOTSTRAP))['bootstrap_project'](PROJECT_DIR)

In [ ]:
import pandas as pd
decisions = pd.read_csv(DECISIONS, encoding='utf-8-sig', keep_default_na=False)
registry = pd.read_csv(REGISTRY, encoding='utf-8-sig', keep_default_na=False)
context = registry[['split', 'document_id', 'entity_id', 'context']].drop_duplicates()
review = decisions.merge(context, on=['split', 'document_id', 'entity_id'], how='left')
display(decisions['decision_status'].value_counts().rename_axis('status').to_frame('count'))
display(review.loc[review.decision_status == 'REVIEW_REQUIRED', ['split','document_id','entity_id','surface','old_type','new_type','reason','context']])

## Изменение решения

Укажите документ и entity ID после просмотра полного контекста. Допустимые статусы: `ACCEPTED`, `REJECTED`, `REVIEW_REQUIRED`.

In [ ]:
from datetime import datetime, timezone
VALID_STATUSES = {'ACCEPTED', 'REJECTED', 'REVIEW_REQUIRED'}

def set_decision(document_id, entity_id, status, reviewer, comment):
    if status not in VALID_STATUSES:
        raise ValueError(f'Недопустимый статус: {status}')
    mask = (decisions.document_id == document_id) & (decisions.entity_id == entity_id)
    if mask.sum() != 1:
        raise ValueError(f'Ожидалась одна строка, найдено {mask.sum()}')
    decisions.loc[mask, 'decision_status'] = status
    decisions.loc[mask, 'reviewer'] = reviewer
    decisions.loc[mask, 'review_comment'] = comment
    decisions.loc[mask, 'reviewed_at'] = datetime.now(timezone.utc).isoformat()

# Пример (раскомментируйте и заполните):
# set_decision('document_id', 'T1', 'ACCEPTED', 'your_name', 'Причина решения')

In [ ]:
SAVE = False  # Поменяйте на True только после завершения текущей порции ревизии.
if SAVE:
    decisions.to_csv(DECISIONS, index=False, encoding='utf-8-sig')
    print('Сохранено:', DECISIONS)
else:
    print('Dry run: файл решений не изменён.')